## 7. Salvar Resultados em Arquivo

Exporta os resultados processados e artefatos gerados para uma pasta de saída, com nomes consistentes para facilitar uso posterior.

In [ ]:
# 7) Salvar Resultados em Arquivo
output_dir = Path(params.output_dir)
output_dir.mkdir(exist_ok=True)

metricas.to_csv(output_dir / 'metricas_execucao.csv', index=False)
resultado.to_csv(output_dir / 'resultado_execucao.csv', index=False)

with open(output_dir / 'resumo_execucao.json', 'w', encoding='utf-8') as f:
    json.dump({
        'n_rows': int(raw_df.shape[0]),
        'grupos': sorted(raw_df['grupo'].unique().tolist()),
        'media_geral': round(float(raw_df['valor'].mean()), 2),
    }, f, ensure_ascii=False, indent=2)

print(f'Arquivos salvos em: {output_dir.resolve()}')
print('Arquivos criados:')
for arquivo in sorted(output_dir.iterdir()):
    print(f' - {arquivo.name}')


## 6. Visualizar Saídas

Cria gráficos e tabelas para inspeção rápida dos resultados e confirmação visual da distribuição e comparação entre grupos.

In [ ]:
# 6) Visualizar Saídas
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribuição de valores
axes[0].hist(raw_df['valor'], bins=20, color='steelblue', edgecolor='black')
axes[0].set_title('Distribuição de valor')
axes[0].set_xlabel('Valor')
axes[0].set_ylabel('Frequência')

# Comparação por grupo
metricas.plot(kind='bar', x='grupo', y='media_valor', ax=axes[1], color='darkorange', edgecolor='black')
axes[1].set_title('Média por grupo')
axes[1].set_xlabel('Grupo')
axes[1].set_ylabel('Média')

plt.tight_layout()
plt.savefig(Path(params.output_dir) / 'visualizacao_execucao.png', dpi=180)
plt.show()

metricas


## 5. Validar Resultados com Asserções

Inclui checagens básicas para tipos, dimensões e faixas esperadas, ajudando a detectar problemas cedo na execução.

In [ ]:
# 5) Validar Resultados com Asserções
assert isinstance(raw_df, pd.DataFrame), 'raw_df deve ser um DataFrame'
assert raw_df.shape[0] == params.n_rows, 'Quantidade de linhas inesperada'
assert raw_df['valor'].between(-1000, 1000).all(), 'Valores fora da faixa esperada'
assert set(raw_df['grupo'].unique()).issubset({'A', 'B', 'C'}), 'Grupos fora dos esperados'
assert metricas['qtd'].sum() == params.n_rows, 'Agregação não soma as linhas originais'
assert resultado['status'].isin(['acima_media', 'abaixo_media']).all(), 'Status com valores inválidos'

print('Validação concluída com sucesso.')
print(metricas)


## 4. Implementar Lógica Principal

Codifica o fluxo central em funções pequenas e testáveis, mantendo a transformação e agregação separadas da etapa de validação.

In [ ]:
# 4) Implementar Lógica Principal
def calcular_metricas(df: pd.DataFrame) -> pd.DataFrame:
    resumo = (
        df.groupby('grupo', as_index=False)
          .agg(media_valor=('valor', 'mean'),
               mediana_valor=('valor', 'median'),
               qtd=('valor', 'size'))
          .sort_values('media_valor', ascending=False)
          .reset_index(drop=True)
    )
    return resumo


def adicionar_status(df: pd.DataFrame) -> pd.DataFrame:
    resultado = df.copy()
    resultado['status'] = np.where(resultado['valor'] >= 100, 'acima_media', 'abaixo_media')
    return resultado

metricas = calcular_metricas(raw_df)
resultado = adicionar_status(raw_df)

metricas.head()


## 3. Carregar ou Gerar Dados de Exemplo

Aqui os dados são gerados com seed fixa para permitir execução imediata e reprodutibilidade em qualquer ambiente do VS Code.

In [ ]:
# 3) Carregar ou Gerar Dados de Exemplo
rng = np.random.default_rng(params.random_seed)

# Dados sintéticos para execução imediata
raw_df = pd.DataFrame({
    'grupo': np.random.choice(['A', 'B', 'C'], size=params.n_rows),
    'valor': rng.normal(loc=100, scale=15, size=params.n_rows),
    'tempo': rng.integers(1, 30, size=params.n_rows),
})

raw_df['valor'] = raw_df['valor'].round(2)
raw_df.head()


## 2. Definir Parâmetros de Execução

Cria uma célula de parâmetros para facilitar ajustes sem precisar editar o restante do notebook.

In [ ]:
# 2) Definir Parâmetros de Execução
from dataclasses import dataclass

@dataclass
class ExecParams:
    output_dir: str = "output"
    random_seed: int = 42
    n_rows: int = 200
    test_mode: bool = True

params = ExecParams()

Path(params.output_dir).mkdir(exist_ok=True)
print(params)


## 1. Configurar Ambiente e Dependências

Importa bibliotecas principais, define opções de exibição e verifica versões para garantir reprodutibilidade.

In [ ]:
# 1) Configurar Ambiente e Dependências
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)


# Notebook de execução prática em Python

Este notebook foi montado como uma estrutura de referência para rodar uma tarefa de ponta a ponta no VS Code, com configuração, processamento, validação e exportação de artefatos.